In [1]:
# Setup for Colab and local environments
import os, sys
from pathlib import Path

# Detect if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    # Clone repo to get dataset
    os.system('git clone https://github.com/nelsunnel/LLMs-and-GenAI-Assignment.git /content/project')
    os.chdir('/content/project')
    # Install requirements
    os.system('pip install -q -r requirements.txt')
    PROJECT_ROOT = Path('/content/project')
else:
    print('Running locally')
    PROJECT_ROOT = Path('.')

print('Project root:', PROJECT_ROOT)

Running locally
Project root: .


# Q3 — CLIP ViT-B/16 zero-shot evaluation

Compute zero-shot per-class precision and recall using CLIP (ViT-B/16).

In [2]:
# Install note: if CLIP is not available, install via pip:
# pip install ftfy regex tqdm
# pip install git+https://github.com/openai/CLIP.git

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import clip
from torchvision import transforms
from sklearn.metrics import precision_recall_fscore_support
from tqdm.notebook import tqdm

DATA_ROOT = PROJECT_ROOT / 'Datasets' / 'dataset'
if not DATA_ROOT.exists(): raise RuntimeError('Dataset not found')
CLASSES = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir()])

def build_image_list(root, classes):
    items = []
    for idx, c in enumerate(classes):
        p = Path(root)/c
        imgs = sorted([x for x in p.iterdir() if x.suffix.lower() in ['.jpg','.jpeg','.png']])
        for im in imgs:
            items.append((str(im), idx))
    return items

items = build_image_list(DATA_ROOT, CLASSES)
print('Total images for zero-shot:', len(items))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, preprocess = clip.load('ViT-B/16', device=device)

# Prepare text prompts
prompts = [f'a photo of a {c.replace("_", " ")}' for c in CLASSES]
text_tokens = clip.tokenize(prompts).to(device)
with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=1, keepdim=True)

# Evaluate images in batches to avoid OOM
batch_size = 32
y_true, y_pred = [], []
for i in tqdm(range(0, len(items), batch_size), desc='Evaluating Batches'):
    batch_items = items[i:i+batch_size]
    imgs = [preprocess(Image.open(p).convert('RGB')) for p, _ in batch_items]
    imgs = torch.stack(imgs).to(device)
    labels = [label for _, label in batch_items]
    
    with torch.no_grad():
        image_features = model.encode_image(imgs)
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        sims = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        preds = sims.argmax(dim=1).cpu().numpy()
    
    y_true.extend(labels)
    y_pred.extend(preds.tolist())

prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(CLASSES))), zero_division=0)
df = pd.DataFrame({'class': CLASSES, 'precision': prec, 'recall': rec, 'f1-score': f1, 'support': sup})
display(df)
print('Zero-shot evaluation complete')

Total images for zero-shot: 805


100%|███████████████████████████████████████| 335M/335M [01:18<00:00, 4.47MiB/s]


Evaluating Batches:   0%|          | 0/26 [00:00<?, ?it/s]

,class,precision,recall,f1-score,support
0,accordion,1.000000,1.000000,1.000000,55
1,bass,1.000000,1.000000,1.000000,54
2,camera,1.000000,1.000000,1.000000,50
3,crocodile,0.755102,0.740000,0.747475,50
4,crocodile_head,0.750000,0.764706,0.757282,51
5,cup,1.000000,1.000000,1.000000,57
6,dollar_bill,1.000000,1.000000,1.000000,52
7,emu,1.000000,1.000000,1.000000,53
8,gramophone,0.980769,1.000000,0.990291,51
9,hedgehog,1.000000,1.000000,1.000000,54


Zero-shot evaluation complete
